# Additional Rigor Experiments — Colab Runner

Run the four council-recommended experiments for IdiomBERT + MultiIdiom on Colab GPU.
Supports two modes:

1. **Sequential** — run all four experiments back-to-back in a single Colab session (~3-4 hours on A100, ~12 hours on T4).
2. **Parallel across sessions** — open this notebook in 4 separate browser tabs and run a different experiment in each session. Best for free Colab where session limits make sequential runs risky. See the *Multi-Session Parallel Plan* section at the bottom.

All outputs save to `MyDrive/IdiomatorRigor/` so they survive runtime disconnects.

## 0. Pick GPU runtime

**Before running:** `Runtime → Change runtime type → A100 GPU` (Pro+) or `T4 GPU` (free). Then run the next cell to confirm.

In [ ]:
!nvidia-smi

## 1. Clone repo and install dependencies

In [ ]:
import os
from pathlib import Path

REPO_DIR = '/content/Idiomator_Research'

if not Path(REPO_DIR).exists():
    !git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git $REPO_DIR

%cd $REPO_DIR/Research_And_Training
!pip install -q -r Requirements.txt
!pip install -q openai python-dotenv  # needed by run_04_llama3_baseline.py

## 2. Mount Google Drive for persistent output

Saves model checkpoints + metrics to Drive so a Colab disconnect doesn't lose hours of training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)

# Symlink models/ → Drive so trainer output writes through to Drive
if not Path('models').is_symlink() and not Path('models').exists():
    !ln -s $DRIVE_OUT models
elif Path('models').exists() and not Path('models').is_symlink():
    # Pre-existing models/ dir in repo — keep it, only redirect rigor_* outputs
    Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
    print('Using models/ as-is. rigor_* outputs will be copied to Drive after each run.')

!ls -la models/ | head -20

## 3. API keys for LLM baseline (experiment 04 only)

Add `GROQ_API_KEY` to **Colab Secrets** (key icon in left sidebar). This is the preprint-grade default — Groq's free tier (14.4k requests/day) is sufficient for the 960-example test set.

Before the ARR Aug 2026 submission, you'll switch to DeepInfra (BF16, ~$0.25 per full run, $5 minimum top-up) and add `DEEPINFRA_API_KEY` then. See the script docstring "Dual-track precision strategy" for the cutover plan. Skip this cell entirely if you're not running experiment 04 in this session.

*Note: experiment 04 doesn't need a GPU — you can run it locally on a laptop instead of Colab.*

In [ ]:
from google.colab import userdata

for key in ['TOGETHER_API_KEY', 'DEEPINFRA_API_KEY', 'GROQ_API_KEY']:
    try:
        os.environ[key] = userdata.get(key)
        print(f'{key}: set')
    except Exception:
        print(f'{key}: not in Colab Secrets (ok unless you need this provider)')

## 4. Pick which experiment to run

Set `RUN` below. Use `'all'` for sequential mode in one session, or one of `'01'`, `'02'`, `'03'`, `'04'` for parallel-across-sessions mode.

| Code | Experiment | A100 | T4 |
|------|------------|------|-----|
| `'01'` | BIO + MuRIL on Telugu (decision-influencing — run first) | ~45 min | ~2.5 hr |
| `'02'` | XLM-R QA-vs-BIO replication (two trainings) | ~90 min | ~5 hr |
| `'03'` | MuRIL Joint on full training | ~50 min | ~2.5 hr |
| `'04'` | Llama-3.3-70B baseline (API, no GPU needed) | ~30 min | ~30 min |
| `'all'` | Sequential — runs 01 → 02 → 03 → 04 | ~3.5 hr | infeasible |

In [ ]:
RUN = '01'  # change to '01', '02', '03', '04', or 'all'

print(f'Will run experiment: {RUN}')

## On hyperparameter tuning — why we don't re-tune per encoder

Tension worth being explicit about:

- **Main paper Sec 7**: hyperparameters were deliberately fixed across all 15 training combinations so that observed differences reflect *training data composition*, not optimization.
- **Reviewer #2 risk**: "You didn't tune XLM-R fairly."

**Design choice taken here:** use each encoder's **published default learning rate** from its release paper. All other hyperparameters (loss weights β=0.3 γ=1.9, batch=32, dropout, epochs) are held identical to the main-paper Joint mBERT system. This isolates the encoder swap from a hyperparameter search.

| Encoder | LR used here | Source |
|---------|-------------|--------|
| mBERT (main paper) | 2e-5 / 3.27e-5 (BIO) | Already established in paper |
| XLM-R | 1e-5 | Conneau et al. 2020 standard |
| MuRIL | 2e-5 | Khanuja et al. 2021 standard, same family as mBERT |

**Paper-side framing**: "We use each encoder's published learning rate; all other hyperparameters fixed to the main-paper Joint mBERT system, following our protocol of fixing optimization to isolate the variable under study."

**If a reviewer pushes back**, run the optional `hp_sweep_xlmr_dev.sh` (3-point LR sweep on dev only) — added separately. Don't run it pre-emptively or the encoder-swap experiment is confounded.

## Experiment 01 — BIO + MuRIL on Telugu (decision-influencing)

Tests whether Telugu BIO=0.0000 is architectural or tokenizer-driven. **This result determines which paper carries the Main shot** (MultiIdiom if BIO recovers with MuRIL, IdiomBERT if it doesn't).

In [ ]:
if RUN in ('01', 'all'):
    !bash Additional_Rigor_Experiments/run_01_bio_muril_recovery.sh

## Experiment 02 — XLM-R replication of QA-vs-BIO

Two trainings: Joint (System E) and BIO (System G) with XLM-R encoder. Tests whether the QA-style-beats-BIO finding survives an encoder swap — the single biggest ceiling-mover for IdiomBERT.

In [ ]:
if RUN in ('02', 'all'):
    !bash Additional_Rigor_Experiments/run_02_xlmr_qa_vs_bio.sh

## Experiment 03 — MuRIL Joint (System E) on full training

Closes the "first Telugu resource but no Indic encoder" gap. Expects MuRIL to help on HI/TE, possibly hurt on ES (acceptable — that asymmetry is the point).

In [ ]:
if RUN in ('03', 'all'):
    !bash Additional_Rigor_Experiments/run_03_muril_joint.sh

## Experiment 04 — Llama-3.3-70B baseline

API-only (no GPU). Defuses the single-LLM (GPT-4o only) reviewer attack. Run while GPU experiments are running for free parallelism.

In [ ]:
if RUN in ('04', 'all'):
    # Groq free tier is the preprint default (LPU-quantized, $0).
    # Before ARR Aug 2026: switch to --provider deepinfra (BF16, ~$0.25)
    # and re-run. See script docstring "Dual-track precision strategy".
    !python Additional_Rigor_Experiments/run_04_llama3_baseline.py \
        --provider groq \
        --test_path idioms_structured/Splits/test.jsonl \
        --output_dir models/rigor_llama3_single

## 5. Inspect results

Each experiment writes `metrics.json` and `test_predictions.jsonl` to its output dir. The metrics JSON shape matches `Ablations/GPT-Baseline.py` so `Evaluation/Full_evaluation.py` consumes it without changes.

In [ ]:
import json, glob

for path in sorted(glob.glob('models/rigor_*/metrics.json')):
    print('\n──', path, '──')
    print(json.dumps(json.load(open(path)), indent=2)[:600])

## 6. Save final outputs to Drive (if symlink wasn't used)

Skip if your `models/` is already a Drive symlink (cell 2 did this). Run if you kept `models/` local and want to preserve `rigor_*` results on Drive.

In [ ]:
if not Path('models').is_symlink():
    !rsync -av models/rigor_* $DRIVE_OUT/
    print(f'Copied rigor_* outputs to {DRIVE_OUT}')
else:
    print('models/ is a Drive symlink — already persisted.')

---

## Multi-Session Parallel Plan

If you want to fan out experiments across multiple free-tier Colab sessions (T4 is slow; parallelism wins back wall-clock time):

**Setup**

1. Open this notebook (`File → Save a copy in Drive`) so you can open it in multiple tabs.
2. Open 4 separate browser windows/incognito sessions, sign into Colab with the same Google account in each.
3. Open the saved notebook in each session. (Each session gets its own free T4.)

**In each session, change one line:**

| Tab | `RUN = ` | Output goes to |
|-----|----------|----------------|
| 1 | `'01'` | `MyDrive/IdiomatorRigor/rigor_bio_muril_full/` |
| 2 | `'02'` | `MyDrive/IdiomatorRigor/rigor_joint_xlmr_full/` + `rigor_bio_xlmr_full/` |
| 3 | `'03'` | `MyDrive/IdiomatorRigor/rigor_joint_muril_full/` |
| 4 | `'04'` | `MyDrive/IdiomatorRigor/rigor_llama3_single/` |

Outputs don't collide — each experiment writes to its own subdir under `MyDrive/IdiomatorRigor/`.

**Caveats for free Colab**

- Free Colab limits total concurrent GPU sessions (usually 1-2 simultaneous, sometimes 3). If you can't open 4, run 01 + 04 in parallel first (04 is API-only and doesn't need GPU), then 02 + 03 next.
- Free sessions disconnect after ~90 min idle or 12 hours total. The Drive symlink in cell 2 means a disconnect at hour 11 doesn't lose the model — checkpointing in the trainer code persists incrementally.
- If you have **Colab Pro+** ($50/mo, A100 access), sequential `RUN = 'all'` in a single session is simpler and finishes in ~3.5 hours.

**Pro tip**: experiment 04 (Llama-3 API) doesn't need a GPU at all. Run it in a **CPU runtime** (Runtime → Change runtime type → None) so it doesn't burn your GPU quota. You can run it concurrently with any GPU experiment for true parallelism.

## Fan-out across multiple people's Colab accounts

If one person doesn't have Pro+ but a group of 3-4 collaborators each have free Colab accounts, you can split the workload across accounts. Each collaborator runs one experiment in their own Colab; results land in a shared Drive folder.

### Owner setup (one-time)

1. In Google Drive, create folder `IdiomatorRigor/`.
2. Right-click → Share → add each collaborator's Google account with **Editor** access.
3. Inside, create a `claims.md` file:
   ```
   Experiment 01 — BIO+MuRIL recovery — CLAIMED BY: ___
   Experiment 02 — XLM-R QA-vs-BIO    — CLAIMED BY: ___
   Experiment 03 — MuRIL Joint         — CLAIMED BY: ___
   Experiment 04 — Llama-3.3 baseline  — CLAIMED BY: ___
   ```
   First collaborator to edit claims an experiment.

### Each collaborator does once

1. Open `IdiomatorRigor/` in Drive web → **Add shortcut to My Drive**. This step is required — Colab cannot write to "Shared with me" directly.
2. Open this notebook in their Colab.
3. Edit `claims.md` to put their name next to an experiment.
4. Set `RUN = '01'` (or whichever they claimed) in cell 4.
5. Run all cells.

### Constraints

- **Experiment 04 doesn't need Colab — owner should run it on their own laptop.** The full single-stage Llama-3.3-70B baseline on the 960-example test set is ~$0.23 on DeepInfra (BF16, paper-grade) and finishes in 5-10 min. There's no reason to involve collaborators or burn a Colab session on it. Total LLM-API budget worst-case (4-shot + 5-seed sweep): ~$1.10.
- **Do not share the DeepInfra API key** with collaborators — it bills per token. Keep all paid-API calls on the owner's machine. If a collaborator wants to smoke-test the script in their own Colab, they can use their own **free Groq account** (Llama-3.3-70B on `--provider groq`, 14.4k requests/day free tier, LPU-quantized — fine for sanity checks, NOT for paper-reported numbers).
- Each free Colab account gets one T4 at a time. 3 collaborators ≈ 3 concurrent T4s for experiments 01-03 — roughly the same wall-clock as one A100 on a single Pro+ account, but free.
- Drive free quota is 15 GB. The 3 model checkpoints together are ~4 GB. Have collaborators delete checkpoints after metrics extract, or upgrade Drive to 100 GB ($2/mo).

### Owner pulls final results

Once all entries in `claims.md` say `complete`, on the owner's laptop:

```bash
rsync -av ~/Google\ Drive/My\ Drive/IdiomatorRigor/ \
    ~/Desktop/Idiomator_Research/Research_And_Training/models/
cd ~/Desktop/Idiomator_Research/Research_And_Training
python Evaluation/Full_evaluation.py
```

## After all experiments finish

Locally on your laptop (not Colab), pull results from Drive and fold into the paper:

```bash
# Sync results from Drive to local repo
rsync -av ~/Google\ Drive/My\ Drive/IdiomatorRigor/ \
    ~/Desktop/Idiomator_Research/Research_And_Training/models/

# Re-run main eval — picks up new rigor_* dirs automatically
cd ~/Desktop/Idiomator_Research/Research_And_Training
python Evaluation/Full_evaluation.py
```

Update the paper drafts:
- IdiomBERT Table 4 — add rows: System E (XLM-R), System E (MuRIL), System G (XLM-R), System G (MuRIL), Llama-3.3 single
- IdiomBERT Section 6.2 — narrate the QA-vs-BIO survival (or non-survival) under XLM-R
- MultiIdiom Section 6 — add Llama-3.3 column to Table 3; if BIO+MuRIL recovers on TE, this becomes the headline of MultiIdiom's contribution